In [16]:
import sys
sys.path.append('..')

In [17]:
import pandas as pd
import torch
from src.load_data import load_data
from sklearn.model_selection import train_test_split
import torch.nn as nn

In [18]:
df = load_data('../data/raw/creditcard.csv')

In [19]:
df['HourOfDay'] = (df['Time'] // 3600) % 24

In [20]:
df_normal = df[df['Class'] == 0]
df_fraud = df[df['Class'] == 1]

In [21]:
X = df_normal[['V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10',
       'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20',
       'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Amount',
    'HourOfDay']]
y = df_normal['Class']

X_train_n, X_test_n, y_train_n, y_test_n = train_test_split(X, y, test_size=0.3, stratify=y,random_state=42)

In [22]:
X_fraud = df_fraud[['V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10',
       'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20',
       'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Amount',
    'HourOfDay']]
y_fraud = df_fraud['Class']

In [23]:
X_test = pd.concat([X_test_n, X_fraud])
y_test = pd.concat([y_test_n, y_fraud])

In [24]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_n)
X_test_scaled = scaler.transform(X_test)

In [25]:
X_train_scaled.shape

(199020, 30)

In [26]:
X_test_scaled.shape

(85787, 30)

In [27]:
X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)

In [28]:
class Autoencoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(30, 16),
            nn.ReLU(),
            nn.Linear(16, 8)
        )
        self.decoder = nn.Sequential(
            nn.Linear(8, 16),
            nn.ReLU(),
            nn.Linear(16, 30)
        )

    def forward(self, x):
        сжатое = self.encoder(x)
        восстановленное = self.decoder(сжатое)
        return восстановленное

In [29]:
model = Autoencoder()
model

Autoencoder(
  (encoder): Sequential(
    (0): Linear(in_features=30, out_features=16, bias=True)
    (1): ReLU()
    (2): Linear(in_features=16, out_features=8, bias=True)
  )
  (decoder): Sequential(
    (0): Linear(in_features=8, out_features=16, bias=True)
    (1): ReLU()
    (2): Linear(in_features=16, out_features=30, bias=True)
  )
)